# Prediction: Run All Systems Once & Save Results

This notebook runs all five extraction systems **once** and saves their outputs to `./predictions/`.  
The evaluation notebook (`evaluation.ipynb`) loads these saved files — no re-training or inference needed.

**Systems:**
- `SVM-TF-IDF` — LinearSVC with TF-IDF features (rule-based, simple pipeline)
- `SVM-SpaCy` — LinearSVC with SpaCy `en_core_web_md` vectors (rule-based, rich features)
- `LLM-1-shot` — Llama 3.1 with 1 in-context example
- `LLM-2-shot` — Llama 3.1 with 2 in-context examples
- `LLM-3-shot` — Llama 3.1 with 3 in-context examples

**Saved files:**
```
predictions/
  gold_test_docs.pkl      ← gold token masks for 184 test documents
  pred_svm_tfidf.pkl      ← binary token masks + sentence confidence scores
  pred_svm_spacy.pkl
  pred_llm_1shot.pkl      ← binary token masks + raw spans + extraction status
  pred_llm_2shot.pkl
  pred_llm_3shot.pkl
```

In [1]:
import os, pickle, warnings
import numpy as np
import pandas as pd
from pathlib import Path
from difflib import SequenceMatcher
from collections import defaultdict
from sklearn.svm import LinearSVC
from sklearn.feature_extraction.text import TfidfVectorizer

warnings.filterwarnings('ignore')
os.makedirs('./predictions', exist_ok=True)
os.makedirs('./data', exist_ok=True)

DATA_DIR  = Path('./ebm_nlp_2_00/documents')
LABEL_DIR = Path('./ebm_nlp_2_00/annotations/aggregated/hierarchical_labels')
PICO_DIRS = {'P': 'participants', 'I': 'interventions', 'O': 'outcomes'}
print('Setup done.')

Setup done.


## Step 1 — Build Gold Test Corpus

In [2]:
GOLD_PATH = Path('./predictions/gold_test_docs.pkl')

def get_test_ids(key):
    d = LABEL_DIR / PICO_DIRS[key] / 'test' / 'gold'
    return sorted(p.stem.split('.')[0] for p in d.glob('*.AGGREGATED.ann'))

def load_tokens(doc_id):
    with open(DATA_DIR / f'{doc_id}.tokens', encoding='utf-8') as f:
        return [l.strip() for l in f]

def load_binary_mask(doc_id, key):
    path = LABEL_DIR / PICO_DIRS[key] / 'test' / 'gold' / f'{doc_id}.AGGREGATED.ann'
    with open(path, encoding='utf-8') as f:
        tags = [l.strip() for l in f]
    return [0 if t == '0' else 1 for t in tags]

if GOLD_PATH.exists():
    with open(GOLD_PATH, 'rb') as f:
        test_docs = pickle.load(f)
    print(f'Loaded gold from cache: {len(test_docs)} docs')
else:
    common_ids = sorted(
        set(get_test_ids('P')) & set(get_test_ids('I')) & set(get_test_ids('O'))
    )
    test_docs = []
    for doc_id in common_ids:
        try:
            tokens = load_tokens(doc_id)
            masks  = {k: load_binary_mask(doc_id, k) for k in ['P', 'I', 'O']}
            if all(len(masks[k]) == len(tokens) for k in masks):
                test_docs.append({'doc_id': doc_id, 'tokens': tokens,
                                  'text': ' '.join(tokens), 'masks': masks})
        except FileNotFoundError:
            pass
    with open(GOLD_PATH, 'wb') as f:
        pickle.dump(test_docs, f)
    print(f'Built and saved gold: {len(test_docs)} docs → {GOLD_PATH}')

Built and saved gold: 184 docs → predictions\gold_test_docs.pkl


## Step 2 — SVM Systems (Train + Predict)

In [3]:
import spacy
print('Loading SpaCy en_core_web_md…')
nlp = spacy.load('en_core_web_md')
print('Ready.')

Loading SpaCy en_core_web_md…
Ready.


In [4]:
def build_sentence_dataset(target_key, split='train'):
    '''Build (texts, vectors, labels) from train or test/gold split.'''
    if split == 'train':
        label_dir   = LABEL_DIR / PICO_DIRS[target_key] / 'train'
        label_files = [f for f in os.listdir(label_dir)
                       if f.endswith('.ann') or f.endswith('.tags')]
        get_path = lambda fname: label_dir / fname
    else:
        label_dir   = LABEL_DIR / PICO_DIRS[target_key] / 'test' / 'gold'
        label_files = [p.name for p in label_dir.glob('*.AGGREGATED.ann')]
        get_path = lambda fname: label_dir / fname

    texts, vectors, labels = [], [], []
    for fname in label_files:
        doc_id   = fname.split('.')[0]
        tok_path = DATA_DIR / f'{doc_id}.tokens'
        if not tok_path.exists(): continue
        with open(tok_path, encoding='utf-8') as f: toks = [l.strip() for l in f]
        with open(get_path(fname), encoding='utf-8') as f: labs = [l.strip() for l in f]
        if len(toks) != len(labs): continue
        cur_t, cur_l = [], []
        for tok, lab in zip(toks, labs):
            cur_t.append(tok); cur_l.append(lab)
            if tok in ('.', '?', '!'):
                if len(cur_t) > 4:
                    sent   = ' '.join(cur_t)
                    counts = {l: cur_l.count(l) for l in cur_l if l not in ('0','NONE','O')}
                    label  = max(counts, key=counts.get) if counts else 'NONE'
                    texts.append(sent)
                    vectors.append(nlp(sent).vector)
                    labels.append(label)
                cur_t, cur_l = [], []
    return texts, np.array(vectors), np.array(labels)

# Build or load training data
svm_train = {}
for key in ['P', 'I', 'O']:
    cache = Path(f'./data/labeled_data_{key}.npz')
    if cache.exists():
        d = np.load(cache, allow_pickle=True)
        svm_train[key] = (list(d['texts']), d['vectors'], d['labels'])
        print(f'[{key}] train cached — {len(d["labels"])} sentences')
    else:
        print(f'[{key}] building train sentences…')
        t, v, l = build_sentence_dataset(key, 'train')
        np.savez_compressed(cache, texts=t, vectors=v, labels=l)
        svm_train[key] = (t, v, l)
        print(f'[{key}] done — {len(l)} sentences')

[P] train cached — 49017 sentences
[I] train cached — 50429 sentences
[O] train cached — 50027 sentences


In [5]:
# Train SVM models
svm_models = {}
for key in ['P', 'I', 'O']:
    texts, vecs, labs = svm_train[key]
    tfidf = TfidfVectorizer()
    clf_a = LinearSVC(class_weight='balanced', max_iter=2000)
    clf_a.fit(tfidf.fit_transform(texts), labs)
    clf_b = LinearSVC(class_weight='balanced', max_iter=2000)
    clf_b.fit(vecs, labs)
    svm_models[key] = {'tfidf': (clf_a, tfidf), 'spacy': (clf_b, None)}
    print(f'[{key}] trained — classes: {list(clf_a.classes_)}')

[P] trained — classes: ['1', '2', '3', '4', 'NONE']
[I] trained — classes: ['1', '2', '3', '4', '5', '6', '7', 'NONE']
[O] trained — classes: ['1', '2', '3', '4', '5', '6', 'NONE']


In [6]:
def split_sentences(tokens):
    sentences, cur, start = [], [], 0
    for i, tok in enumerate(tokens):
        cur.append(tok)
        if tok in ('.', '?', '!'):
            if len(cur) > 4:
                sentences.append((start, i, list(cur)))
            cur, start = [], i + 1
    if cur: sentences.append((start, len(tokens) - 1, cur))
    return sentences


def predict_svm(doc, key, strategy):
    '''
    Returns:
      mask   : binary token mask
      scores : list of (start, end, confidence) per sentence
    '''
    clf, tfidf_vec = svm_models[key][strategy]
    classes  = list(clf.classes_)
    none_idx = classes.index('NONE') if 'NONE' in classes else -1

    mask    = [0] * len(doc['tokens'])
    scores  = []
    for start, end, sent_toks in split_sentences(doc['tokens']):
        sent = ' '.join(sent_toks)
        vec  = tfidf_vec.transform([sent]) if strategy == 'tfidf' \
               else nlp(sent).vector.reshape(1, -1)
        raw  = clf.decision_function(vec)[0]
        pred = clf.classes_[np.argmax(raw)]
        pico = [s for i, s in enumerate(raw) if i != none_idx]
        conf = (max(pico) if pico else -99) - (raw[none_idx] if none_idx >= 0 else 99)
        scores.append((start, end, float(conf)))
        if pred != 'NONE':
            for idx in range(start, min(end + 1, len(mask))):
                mask[idx] = 1
    return mask, scores


for strategy, out_path in [('tfidf', './predictions/pred_svm_tfidf.pkl'),
                            ('spacy', './predictions/pred_svm_spacy.pkl')]:
    print(f'Predicting SVM-{strategy}…')
    masks_all  = {}
    scores_all = {}
    for doc in test_docs:
        did = doc['doc_id']
        masks_all[did]  = {}
        scores_all[did] = {}
        for key in ['P', 'I', 'O']:
            m, s = predict_svm(doc, key, strategy)
            masks_all[did][key]  = m
            scores_all[did][key] = s

    payload = {
        'system':   f'SVM-{strategy.upper()}',
        'masks':    masks_all,
        'scores':   scores_all,   # for coverage–precision curves
    }
    with open(out_path, 'wb') as f:
        pickle.dump(payload, f)
    print(f'  Saved → {out_path}')

Predicting SVM-tfidf…
  Saved → ./predictions/pred_svm_tfidf.pkl
Predicting SVM-spacy…
  Saved → ./predictions/pred_svm_spacy.pkl


## Step 3 — LLM Systems (Load CSVs + Convert Spans)

In [7]:
def span_to_mask(doc_text, span_str, n_tokens, threshold=0.6):
    '''
    Locate span_str in doc_text via sliding-window SequenceMatcher.
    Returns (mask, status) where status is 'abstain' | 'hallucination' | 'located'.
    '''
    mask = [0] * n_tokens
    if not span_str or str(span_str).lower() in ('null', 'none', 'nan', ''):
        return mask, 'abstain'

    span_str   = str(span_str)
    orig_words = doc_text.split()
    ext_words  = span_str.split()
    window     = max(1, len(ext_words))

    best_ratio, best_start = 0.0, 0
    for i in range(max(1, len(orig_words) - window + 1)):
        chunk = ' '.join(orig_words[i: i + window])
        r     = SequenceMatcher(None, span_str.lower(), chunk.lower()).ratio()
        if r > best_ratio:
            best_ratio, best_start = r, i

    if best_ratio < threshold:
        return mask, 'hallucination'

    for j in range(window):
        if best_start + j < n_tokens:
            mask[best_start + j] = 1
    return mask, 'located'


def process_llm_csv(csv_path, test_docs, threshold=0.6):
    df_csv      = pd.read_csv(csv_path)
    text_lookup = {d['text'][:150]: d for d in test_docs}
    COLS        = {'P': 'Pop_Extracted', 'I': 'Int_Extracted', 'O': 'Out_Extracted'}

    masks_all   = {}
    spans_all   = {}    # raw span strings  (for threshold sweeping in eval)
    status_all  = {}
    matched = 0

    for _, row in df_csv.iterrows():
        rk  = str(row.get('Text', ''))[:150]
        doc = text_lookup.get(rk)
        if doc is None:
            best_r, best_doc = 0.0, None
            for k, d in text_lookup.items():
                r = SequenceMatcher(None, rk, k).ratio()
                if r > best_r: best_r, best_doc = r, d
            if best_r > 0.85: doc = best_doc
        if doc is None: continue

        matched += 1
        did  = doc['doc_id']
        ntok = len(doc['tokens'])
        masks_all[did]  = {}
        spans_all[did]  = {}
        status_all[did] = {}

        for key, col in COLS.items():
            raw_span = str(row.get(col, ''))
            mask, status = span_to_mask(doc['text'], raw_span, ntok, threshold)
            masks_all[did][key]  = mask
            spans_all[did][key]  = raw_span
            status_all[did][key] = status

    print(f'  Matched {matched}/{len(df_csv)} rows')
    return masks_all, spans_all, status_all


LLM_CSVS = [
    ('1-shot', 'pico_1_shot_results.csv', './predictions/pred_llm_1shot.pkl'),
    ('2-shot', 'pico_2_shot_results.csv', './predictions/pred_llm_2shot.pkl'),
    ('3-shot', 'pico_3_shot_results.csv', './predictions/pred_llm_3shot.pkl'),
]

for shots, csv_path, out_path in LLM_CSVS:
    print(f'Processing LLM-{shots} from {csv_path}…')
    if not os.path.exists(csv_path):
        print(f'  NOT FOUND — skipping'); continue
    masks, spans, status = process_llm_csv(csv_path, test_docs)
    payload = {
        'system':  f'LLM-{shots}',
        'masks':   masks,
        'spans':   spans,    # raw strings → threshold sweep in eval
        'status':  status,   # abstain / hallucination / located
    }
    with open(out_path, 'wb') as f:
        pickle.dump(payload, f)
    print(f'  Saved → {out_path}')

Processing LLM-1-shot from pico_1_shot_results.csv…
  Matched 184/184 rows
  Saved → ./predictions/pred_llm_1shot.pkl
Processing LLM-2-shot from pico_2_shot_results.csv…
  Matched 184/184 rows
  Saved → ./predictions/pred_llm_2shot.pkl
Processing LLM-3-shot from pico_3_shot_results.csv…
  Matched 184/184 rows
  Saved → ./predictions/pred_llm_3shot.pkl


## Step 4 — Prediction Summary

In [8]:
import os
print('Saved prediction files:')
for p in sorted(Path('./predictions').glob('*.pkl')):
    size_kb = p.stat().st_size / 1024
    with open(p, 'rb') as f:
        obj = pickle.load(f)
    n_docs = len(obj.get('masks', obj) if isinstance(obj, dict) else obj)
    print(f'  {p.name:<30} {size_kb:>8.1f} KB   docs={n_docs}')

print('\nPrediction complete. Run evaluation.ipynb next.')

Saved prediction files:
  gold_test_docs.pkl                983.5 KB   docs=184
  pred_llm_1shot.pkl                354.2 KB   docs=184
  pred_llm_2shot.pkl                344.8 KB   docs=184
  pred_llm_3shot.pkl                372.3 KB   docs=184
  pred_svm_spacy.pkl                394.5 KB   docs=184
  pred_svm_tfidf.pkl                394.5 KB   docs=184

Prediction complete. Run evaluation.ipynb next.
